# Lab 15-12 · PLLs, Regulators and Thermal Management

Six parts, each built on `loops.py`. **Predict first**, then run.

In [1]:
import math
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import loops as lp
from specs import step_specs   # importable because loops.py adds labs/15-03-lab to the path

Icp, Kv, N = 100e-6, 2 * math.pi * 100e6, 100

## Part 1 · Design the PLL's loop filter

*Predict:* if the components are rounded to 13.5 kΩ, 220 pF and 17 pF, does the 60° margin survive?

In [2]:
d = lp.pll_design(200e3, 60, Icp, Kv, N)
print({k: f"{v:.4g}" for k, v in d.items()})
num, den = lp.pll_loop(Icp, Kv, N, d["R"], d["C"], d["C2"])
m = lp.margins(num, den, w_lo=1e2, w_hi=1e10)
print(f"exact parts:   crossover {m['wc'] / 2 / math.pi / 1e3:.1f} kHz, PM {m['pm']:.2f} deg")
num_r, den_r = lp.pll_loop(Icp, Kv, N, 13.5e3, 220e-12, 17e-12)
mr = lp.margins(num_r, den_r, w_lo=1e2, w_hi=1e10)
print(f"rounded parts: crossover {mr['wc'] / 2 / math.pi / 1e3:.1f} kHz, PM {mr['pm']:.2f} deg")
f = np.logspace(3, 8, 2000)
L = lp.freq_resp(num, den, 2 * math.pi * f)
fig, (a1, a2) = plt.subplots(2, 1, sharex=True, figsize=(6, 4.5))
a1.semilogx(f, 20 * np.log10(abs(L))); a1.axhline(0, color="gray"); a1.set_ylabel("|L| (dB)")
a2.semilogx(f, np.degrees(np.unwrap(np.angle(L))) - 360 * (np.degrees(np.angle(L[0])) > 0)); a2.axhline(-180, color="gray")
a2.set_ylabel("phase (deg)"); a2.set_xlabel("f (Hz)")
plt.show()

{'b': '13.93', 'x': '3.732', 'tz': '2.97e-06', 'tp': '2.132e-07', 'K': '4.231e+11', 'Ctot': '2.363e-10', 'C': '2.194e-10', 'C2': '1.697e-11', 'R': '1.354e+04'}
exact parts:   crossover 200.0 kHz, PM 60.00 deg
rounded parts: crossover 199.5 kHz, PM 60.01 deg


## Part 2 · Close the loop and hop the frequency

In [3]:
nT, dT = lp.closed(num, den)
x = d["x"]; wc = 2 * math.pi * 200e3
print("poles (numerical):", np.round(np.sort(np.roots(dT).real), 1))
zeta = (x - 1) / 2
print("poles (factorization):", np.round(sorted([-wc, wc * (-zeta - math.sqrt(zeta**2 - 1)), wc * (-zeta + math.sqrt(zeta**2 - 1))]), 1))
pk, fpk = lp.peaking(nT, dT)
print(f"jitter-transfer peak {pk:.2f} dB at {fpk / 1e3:.1f} kHz; -3 dB bandwidth {lp.bw3db(nT, dT) / 1e3:.0f} kHz")
num1, den1 = lp.pll_loop(Icp, Kv, 101, d["R"], d["C"], d["C2"])
s = step_specs(*lp.closed(num1, den1), 200e-6, band=1e-4, n=400001)
dw = 2 * math.pi * 10e6 * (1 - 100 / 101)
e_pk, t_pk = lp.phase_error_peak(num1, den1, dw, 50e-6)
print(f"hop 1.00 -> 1.01 GHz: overshoot {100 * s['overshoot']:.1f}%, settles within 1 kHz in {s['settle'] * 1e6:.1f} us, "
      f"peak PFD error {e_pk:.3f} rad at {t_pk * 1e6:.2f} us")
if shutil.which("ngspice"):
    out = subprocess.run(["ngspice", "-b", "pll_phase.cir"], capture_output=True, text=True, timeout=120).stdout
    print("ngspice:", [l for l in out.splitlines() if l.startswith(("pk ", "fpk", "f3 "))])
else:
    print("ngspice not installed: skip (CI runs pll_phase.cir through tools/run_spice.py)")

poles (numerical): [-2886030.7 -1256637.1  -547165.6]
poles (factorization): [-2886030.7 -1256637.1  -547165.6]
jitter-transfer peak 1.70 dB at 93.1 kHz; -3 dB bandwidth 313 kHz


hop 1.00 -> 1.01 GHz: overshoot 18.9%, settles within 1 kHz in 17.3 us, peak PFD error 0.412 rad at 1.37 us
ngspice: ['pk                  =  1.703472e+00 at=  9.311079e+04', 'fpk                 =  9.311079e+04 with=  1.703472e+00', 'f3                  =  3.128329e+05']


## Part 3 · Choose the bandwidth for minimum jitter

*Predict:* where does the jitter minimum sit relative to the 316 kHz noise crossing?

In [4]:
fcs = np.logspace(4.5, 6.5, 21)
jit = [lp.jitter(fc)[1] * 1e15 for fc in fcs]
for fc, j in zip(fcs[::4], jit[::4]):
    print(f"fc = {fc / 1e3:7.1f} kHz: {j:6.1f} fs")
plt.semilogx(fcs, jit, "o-"); plt.axvline(10**5.5, color="gray")
plt.xlabel("crossover (Hz)"); plt.ylabel("rms jitter (fs)")
plt.show()

fc =    31.6 kHz:  907.7 fs
fc =    79.4 kHz:  600.7 fs
fc =   199.5 kHz:  434.7 fs
fc =   501.2 kHz:  434.8 fs
fc =  1258.9 kHz:  601.9 fs
fc =  3162.3 kHz:  930.9 fs


## Part 4 · The LDO's ESR window

*Predict:* the margin with a 5 mΩ ceramic capacitor.

In [5]:
esr = np.logspace(-3, 1, 81)
pms = [lp.ldo_pm(r) for r in esr]
lo, hi, rbest, pmbest = lp.esr_window(45)
print(f"45 deg window: {lo:.3f} to {hi:.3f} ohm; best {pmbest:.1f} deg at {rbest:.3f} ohm; 5 mohm: {lp.ldo_pm(0.005):.1f} deg")
plt.semilogx(esr, pms); plt.axhline(45, color="gray"); plt.axvspan(lo, hi, alpha=0.15)
plt.xlabel("ESR (ohm)"); plt.ylabel("phase margin (deg)")
plt.show()
if shutil.which("ngspice"):
    out = subprocess.run(["ngspice", "-b", "ldo_esr.cir"], capture_output=True, text=True, timeout=120).stdout
    print("ngspice:", [l for l in out.splitlines() if l.startswith(("fc ", "pm "))])
else:
    print("ngspice not installed: skip (CI runs ldo_esr.cir through tools/run_spice.py)")

45 deg window: 0.094 to 0.568 ohm; best 57.1 deg at 0.196 ohm; 5 mohm: -4.3 deg
ngspice: ['fc                  =  6.208094e+05', 'pm = 5.434870e+01']


## Part 5 · Droop: load line against PI

In [6]:
dI, C, dV = 20.0, 1e-3, 0.04
t = np.linspace(0, 60e-6, 601)
wc_ll = dI / (C * dV)
wc_pi = (2 / math.e) * dI / (C * dV)
v_ll = lp.droop(dI, C, wc_ll, 0.0, t)
v_pi = lp.droop(dI, C, wc_pi, wc_pi / 4, t)
print(f"load line: crossover {wc_ll / 2 / math.pi / 1e3:.1f} kHz, final {v_ll[-1] * 1e3:.1f} mV")
print(f"PI:        crossover {wc_pi / 2 / math.pi / 1e3:.1f} kHz, peak {v_pi.min() * 1e3:.1f} mV at {t[np.argmin(v_pi)] * 1e6:.2f} us")
plt.plot(t * 1e6, v_ll * 1e3, label="load line (P)"); plt.plot(t * 1e6, v_pi * 1e3, label="critically damped PI")
plt.xlabel("time (us)"); plt.ylabel("deviation (mV)"); plt.legend()
plt.show()

load line: crossover 79.6 kHz, final -40.0 mV
PI:        crossover 58.5 kHz, peak -40.0 mV at 5.40 us


## Part 6 · Turbo, then throttle

*Predict:* how much do halving and doubling the PI gains change the power envelope?

In [7]:
for scale in (0.5, 1.0, 2.0):
    t, TJ, P, t_thr = lp.throttle(10.0 * scale, 10.0 / 0.75 * scale)
    k60 = int(round((t_thr + 60) / 0.1))
    print(f"gains x{scale}: throttling from {t_thr:.1f} s, peak {TJ.max():.2f} C, power 60 s later {P[k60]:.1f} W, final {P[-1]:.1f} W")
    plt.plot(t, P, label=f"gains x{scale}")
tt = np.linspace(75.5, 400, 300)
plt.plot(tt, 130 + 70 * np.exp(-(tt - 75.5) / 31.5), "k--", label="130 + 70 exp(-t'/31.5)")
plt.xlabel("time (s)"); plt.ylabel("permitted power (W)"); plt.legend()
plt.show()

gains x0.5: throttling from 74.8 s, peak 95.30 C, power 60 s later 140.3 W, final 130.0 W
gains x1.0: throttling from 74.8 s, peak 95.15 C, power 60 s later 140.3 W, final 130.0 W
gains x2.0: throttling from 74.8 s, peak 95.07 C, power 60 s later 140.4 W, final 130.0 W
